__This will be a walkthrough of uploading sample data to the data management system for each of the five task types.__

1. Single Label Classification
2. Muli-Label Classification
3. Object detection
4. Semantic Segmentation
5. Instance Segmentation

Note, these are not training samples, they are examples on how to ingest data into the infrastructure described in the README. It shows off the expected input data format (consistent with Ground Truth output) and allows you to test the infrastructure for different label types. While I did create a logging client that returns logs in pandas DataFrame format (and show it off), I highly recommend simply using Athena in the AWS console for log and table visualization. The same holds true for step function flow to detect any failure points.

The following imports the programmatic API main entry point for interacting with the app. Everything is accessible through this client. Before this step, we must login to our account and gain credentials and permissions to interact with the infrastructure. Set up a user in identity center assigned to the account containing the CDK app, grant necessary permissions, and login in the terminal to gain temporary credentials. Ensure the profile you are using is in te local AWS conig file and points to the appropriate account, role, and AWS region.

In [ ]:
# This is the name of the CDK app you deployed.
app_name = "cvdmsv1"

# This is your profile name (see the local aws config file).
profile_name = "developers_admin"

# Login to AWS. This will redirect you to a login screen or be approved if credentials are still valid
# from your previous login.
!aws sso login --profile {profile_name}

In [11]:
# Instantiate the client.
from cvdms_platform import CvdmsApp

app = CvdmsApp(app_name=app_name,
               profile_name=profile_name)

2025-12-28 19:10:46,313 - INFO - Instantiating user instance.
2025-12-28 19:10:46,373 - INFO - Loading cached SSO token for myDefaultSession
2025-12-28 19:10:48,712 - INFO - Using AWS profile: developers_admin, user = developer_brian, region: us-east-1, passed config loading step.
2025-12-28 19:10:48,754 - INFO - Instantiation complete.


In [9]:
manifest_path = r"samples/single_label/single_label_truth_output.manifest"
label_type = "single-label"
job_summary = "Upload some sample COCO images for single label label type."
data_source = "cocoval2017"

upload_info = app.start_upload_job(manifest_path,
                                   label_type,
                                   job_summary = job_summary,
                                   data_source = data_source)

2025-12-28 19:02:38,228 - INFO - Acquiring lock for lock global, new potential holder = 02762d74-b521-48d0-8473-bfa7211832a5, used lock table name cvdmsv1-StorageStack-LockTableB9DACF42-GHYNWC67YSTN
2025-12-28 19:02:38,671 - INFO - Acquired lock: 02762d74-b521-48d0-8473-bfa7211832a5
2025-12-28 19:02:38,741 - INFO - Created job row in job table for IMAGE_UPLOAD event and is status: PENDING.
2025-12-28 19:02:38,745 - INFO - Manifest validated. Uploading to S3...
2025-12-28 19:02:39,231 - INFO - Upload of manifest success: s3://cvdmsv1-storagestack-s3filebucketab18cd0f-biymmmdydpy0/temp/image-upload/02762d74-b521-48d0-8473-bfa7211832a5/02762d74-b521-48d0-8473-bfa7211832a5.manifest
2025-12-28 19:02:39,360 - INFO - Upload of job.json success: s3://cvdmsv1-storagestack-s3filebucketab18cd0f-biymmmdydpy0/temp/image-upload/02762d74-b521-48d0-8473-bfa7211832a5/job.json
2025-12-28 19:02:39,361 - INFO - Done uploading manifest and job.json to S3.
2025-12-28 19:02:39,425 - INFO - Successfully updat

In [10]:
job_id = upload_info.get('job_id')
upload_error = upload_info.get('error')

if job_id:
    print(f"Upload start was a success, job id is {job_id}")
else:
    print(f"Upload start was a failure, error is {upload_error}, see log file in cvdms_platform/api_logs")

Upload start was a success, job id is 02762d74-b521-48d0-8473-bfa7211832a5


Use the log interface to see the logs for this job id stored in a pandas DataFrame. It can take some time for logs to show up.

In [ ]:
log_info = app.get_logs_by_job_id(job_id)

if not log_info.get('error'):
    log_df = log_info['logs_df']
    print('First 10 logs are:')
    print(log_df.head(10))
else:
    log_retrieval_error = log_info.get('error')
    print(f'Error getting logs: {log_retrieval_error}')